# ELT Clientes — Ingesta, Validación y Carga a Delta Table

**Asignatura:** Ingeniería de Datos  
**Docente:** Ing. Sergio Orozco

Este notebook implementa un pipeline **ELT** (Extract → Load → Transform) completo para el dataset de clientes,
siguiendo la arquitectura **Medallion** con capas Bronze y Silver.

| Sección | Tema |
|---------|------|
| **1** | Instalación de dependencias |
| **2** | Importación de librerías y constantes |
| **3 — PARTE A** | Ingesta Bronze: leer CSV y guardar como Parquet |
| **4 — PARTE B** | Transformación: validación y separación de datos limpios/inválidos |
| **5 — PARTE C** | Carga Silver: upsert en Delta Table |
| **6** | Resumen del pipeline |

**Tecnologías:** `pandas`, `pyarrow`, `deltalake`

---

> **¿Qué es la arquitectura Medallion?**  
> Es un patrón de organización de datos en tres capas:  
> 🥉 **Bronze** → datos crudos tal como llegan de la fuente.  
> 🥈 **Silver** → datos limpios y validados, listos para el análisis.  
> 🥇 **Gold** → agregaciones y métricas de negocio (no implementada en este notebook).

## 1. Instalación de Dependencias

Instalamos las librerías necesarias que no vienen con Python por defecto:

- **`pyarrow`**: motor para leer y escribir archivos Parquet (formato columnar eficiente).
- **`deltalake`**: librería Python pura para trabajar con Delta Tables sin necesidad de PySpark.

## 2. Importación de Librerías y Constantes

Definimos en una sola celda todas las rutas, parámetros y mensajes de error que usaremos
a lo largo del notebook.

> **Buena práctica:** centralizar las constantes al inicio facilita el mantenimiento.
> Si mañana cambia el nombre del archivo fuente, solo hay que modificar un lugar.

In [ ]:
import re                          # Expresiones regulares para validar email, teléfono, etc.
import pyarrow as pa               # Motor Parquet y formato intermedio para Delta Table
import pandas as pd                # Manipulación de DataFrames
from pathlib import Path           # Manejo de rutas de archivos de forma multiplataforma
from datetime import datetime      # Para obtener la fecha actual de ingesta
from deltalake import DeltaTable, write_deltalake  # Delta Table sin PySpark
                  # Para manejo de valores nulos y operaciones numéricas

# ── Rutas de trabajo ──────────────────────────────────────────────────────────
# Usamos Path() para que las rutas funcionen tanto en Windows como en Linux/Mac
DIR_BASE   = Path("datos")
DIR_INPUT  = DIR_BASE / "input"
DIR_OUTPUT = DIR_BASE / "output"
DIR_SILVER = DIR_BASE / "silver"

# Creamos las carpetas si no existen (exist_ok=True evita error si ya existen)
DIR_OUTPUT.mkdir(parents=True, exist_ok=True)
DIR_SILVER.mkdir(parents=True, exist_ok=True)

# ── Parámetros de ejecución ───────────────────────────────────────────────────
# La fecha de ingesta se incluye en el nombre del archivo Bronze para tener
# un registro histórico de cada carga.
FECHA_INGESTA  = datetime.today().strftime("%Y%m%d")

ARCHIVO_FUENTE = DIR_INPUT / "clientes_crudos_2.csv"
BRONZE_PATH    = DIR_OUTPUT / f"bronze_clientes_{FECHA_INGESTA}.parquet"
INVALIDOS_PATH = DIR_OUTPUT / f"invalidos_clientes_{FECHA_INGESTA}.parquet"
DELTA_PATH     = str(DIR_SILVER / "clientes_delta")

# ── Diccionario de errores de validación ──────────────────────────────────────
# Centralizamos todos los mensajes de error en un diccionario.
# Esto garantiza que los mensajes sean consistentes y fáciles de mantener.
VALIDATION_ERRORS = {
    "nombres_invalido"    : "nombres contiene dígitos o está vacío",
    "apellidos_invalido"  : "apellidos contiene dígitos o está vacío",
    "documento_invalido"  : "numero_documento nulo, no numérico o fuera del rango 6-8 dígitos",
    "fecha_alta_invalida" : "fecha_alta es nula o no tiene formato de fecha válido",
    "email_invalido"      : "email presente pero no tiene formato válido",
    "telefono_invalido"   : "nro_telefono presente pero no tiene formato válido",
    "sin_metodo_contacto" : "no tiene email ni teléfono con formato válido",
}

print("=== CONSTANTES INICIALIZADAS ===")
print(f"  Fecha de ingesta  : {FECHA_INGESTA}")
print(f"  Fuente            : {ARCHIVO_FUENTE}")
print(f"  Bronze destino    : {BRONZE_PATH}")
print(f"  Inválidos destino : {INVALIDOS_PATH}")
print(f"  Delta Table path  : {DELTA_PATH}")

---

## 3 — PARTE A: Ingesta Bronze

### ¿Qué hacemos?
Leemos el archivo CSV crudo **sin aplicar ninguna transformación** y lo guardamos
como archivo Parquet con la fecha de ejecución en el nombre.

### ¿Por qué guardamos en Parquet?
El formato **Parquet** es columnar, comprimido y mucho más eficiente que CSV para
análisis de datos. Es el estándar en pipelines de datos modernos.

### ¿Por qué usamos `dtype=str`?
Al leer con `dtype=str`, todos los valores se tratan como texto.  
Esto es fundamental para preservar el cero inicial en documentos como `"01234567"`:  
si Pandas lo interpretara como número, lo convertiría a `1234567` (¡perderíamos el cero!).

### Resultado esperado
- Un archivo `bronze_clientes_YYYYMMDD.parquet` en `datos/output/`
- El DataFrame tiene **38 filas** y las mismas columnas del CSV + `fecha_ingesta`

In [ ]:
# ── Lectura del CSV crudo ─────────────────────────────────────────────────────
# dtype=str: fuerza que TODAS las columnas se lean como texto (string).
# Esto evita que Pandas "adivine" tipos y pierda información (ej: ceros iniciales).
df_bronze = pd.read_csv(ARCHIVO_FUENTE, dtype=str, encoding="utf-8")

In [ ]:
df_bronze

In [ ]:
# Reemplazamos las celdas vacías ("") por None (valor nulo real de Python).
# En el CSV, una celda vacía se lee como string vacío "", no como NaN/None.
df_bronze = df_bronze.replace("", None)

# Agregamos la fecha de ingesta como columna para tener trazabilidad.
# Esto nos permite saber cuándo fue cargado cada registro.
df_bronze["fecha_ingesta"] = FECHA_INGESTA

print("=== BRONZE — INGESTA ===")
print(f"  Filas x Columnas : {df_bronze.shape}")
print(f"\n  Tipos de dato:")
print(df_bronze.dtypes)

# ── Guardado en formato Parquet ───────────────────────────────────────────────
# engine="pyarrow": usamos pyarrow como motor de escritura (el más estable).
# index=False: no guardamos el índice del DataFrame (no lo necesitamos).
df_bronze.to_parquet(BRONZE_PATH, engine="pyarrow", index=False)
print(f"\n✅ Bronze guardado en: {BRONZE_PATH}")

In [ ]:
del df_bronze

---

## 4 — PARTE B: Transformación y Validación

En esta sección aplicamos las **reglas de calidad de datos** y separamos los registros
en dos grupos:

- ✅ **`df_validos`**: registros que pasaron todas las validaciones → irán a Silver.
- ❌ **`df_invalidos`**: registros con al menos un problema → se guardan aparte con el motivo.

El proceso tiene cinco pasos (B1 a B5):

| Paso | Descripción |
|------|-------------|
| **B1** | Cargar el Bronze Parquet |
| **B2** | Correcciones silenciosas (fechas opcionales con formato inválido → `NaT`) |
| **B3** | Definir la función de validación `validar_registro()` |
| **B4** | Aplicar la validación y separar los DataFrames |
| **B5** | Guardar los registros inválidos en Parquet |
| **B6** | Guardar en Silver como Delta Table |

### B1 — Carga desde Bronze

Leemos el archivo Parquet que generamos en la Parte A.  
Trabajamos sobre una **copia** (`df_raw.copy()`) para no modificar el DataFrame original
en caso de querer volver a consultarlo más adelante.

In [ ]:
# ── Carga del archivo Bronze ──────────────────────────────────────────────────
df_raw = pd.read_parquet(BRONZE_PATH, engine="pyarrow")

# Trabajamos siempre sobre una copia para preservar los datos originales
df_trabajo = df_raw.copy()

print(f"=== SILVER — CARGA DESDE BRONZE ===")
print(f"  Filas x Columnas: {df_trabajo.shape}")
print(f"\n  Primeras filas:")
df_trabajo

### B2 — Correcciones Silenciosas

Algunos campos de fecha son **opcionales** (`fecha_nacimiento`, `fecha_baja`).  
Si el valor existe pero no tiene un formato de fecha válido, lo reemplazamos por
`NaT` (Not a Time, el equivalente a "nulo" para fechas en Pandas).

> **Importante:** esta corrección es "silenciosa" porque **no mueve el registro a inválidos**.  
> El alumno puede no tener fecha de nacimiento registrada y eso está permitido.  
> El error se registraría solo si el campo fuera obligatorio.

`errors='coerce'` es la clave: en lugar de lanzar un error cuando el formato no es válido,
Pandas simplemente coloca `NaT` en esa celda.

In [ ]:
# ── Corrección silenciosa: fecha_nacimiento ───────────────────────────────────
# Si el valor no es parseable como fecha, queda NaT (nulo de fecha).
# El registro NO se mueve a inválidos por esto.
df_trabajo["fecha_nacimiento"] = pd.to_datetime(
    df_trabajo["fecha_nacimiento"], errors="coerce"
)

df_trabajo["fecha_alta"] = pd.to_datetime(
    df_trabajo["fecha_alta"], errors="coerce"
)

# ── Corrección silenciosa: fecha_baja ─────────────────────────────────────────
# Misma lógica: puede ser nula o tener un formato inválido → NaT, sin invalidar.
df_trabajo["fecha_baja"] = pd.to_datetime(
    df_trabajo["fecha_baja"], errors="coerce"
)

print("=== CORRECCIONES SILENCIOSAS APLICADAS ===")
print(f"  fecha_nacimiento — valores NaT : {df_trabajo['fecha_nacimiento'].isna().sum()}")
print(f"  fecha_alta       — valores NaT : {df_trabajo['fecha_alta'].isna().sum()}")
print(f"  fecha_baja       — valores NaT : {df_trabajo['fecha_baja'].isna().sum()}")

### B3 — Función de Validación

Definimos la función `validar_registro()` que recibe **una fila** del DataFrame
y devuelve una lista con los **códigos de error** encontrados.

Si la lista está vacía → el registro es válido.  
Si tiene uno o más elementos → el registro es inválido.

**Reglas implementadas:**

| # | Campo | Condición para ser inválido |
|---|-------|-----------------------------|
| 1 | `nombres` | Contiene dígitos o está vacío/nulo |
| 2 | `apellidos` | Contiene dígitos o está vacío/nulo |
| 3 | `numero_documento` | Nulo, o no es solo dígitos, o tiene menos de 6 o más de 8 caracteres |
| 4 | `fecha_alta` | Nula o no tiene formato de fecha válido |
| 5 | `email` | Presente pero no tiene el formato `algo@dominio.ext` |
| 6 | `nro_telefono` | Presente pero no es `+` opcional seguido de 7–15 dígitos |
| 7 | Contacto | No tiene ni email ni teléfono válidos (debe tener al menos uno) |

In [ ]:
def validar_registro(fila: pd.Series) -> list:
    """
    Evalúa si una fila del DataFrame de clientes cumple con las reglas de calidad.

    Aplica 7 reglas de validación en orden. Para cada regla que falla, agrega
    la clave correspondiente de VALIDATION_ERRORS a la lista de errores.

    Parámetros
    ----------
    fila : pd.Series
        Una fila del DataFrame de clientes (proviene de df.apply(..., axis=1)).

    Retorna
    -------
    list
        Lista de claves de VALIDATION_ERRORS con los errores encontrados.
        Si la lista está vacía, el registro es válido.

    Ejemplo
    -------
    errores = validar_registro(df_trabajo.iloc[14])
    # Resultado: ['documento_invalido'] para el cliente sin numero_documento
    """
    errores = []

    # ── Regla 1: nombres ──────────────────────────────────────────────────────
    # Convertimos a string y eliminamos espacios para evitar falsos positivos.
    # Un nombre válido no debe contener dígitos ni estar vacío.
    nombres = str(fila.get("nombres") or "").strip()
    if not nombres or bool(re.search(r"\d", nombres)):
        errores.append("nombres_invalido")

    # ── Regla 2: apellidos ────────────────────────────────────────────────────
    # Misma lógica que nombres.
    apellidos = str(fila.get("apellidos") or "").strip()
    if not apellidos or bool(re.search(r"\d", apellidos)):
        errores.append("apellidos_invalido")

    # ── Regla 3: numero_documento ─────────────────────────────────────────────
    # \d{6,8} significa: solo dígitos, mínimo 6, máximo 8 caracteres.
    # fullmatch valida que TODO el string cumpla el patrón (no solo una parte).
    doc = str(fila.get("numero_documento") or "").strip()
    if not doc or not re.fullmatch(r"\d{6,8}", doc):
        errores.append("documento_invalido")

    # ── Regla 4: fecha_alta ───────────────────────────────────────────────────
    # fecha_alta es OBLIGATORIA. Si es nula o no parseable, es un error.
    fecha_alta_raw = fila.get("fecha_alta")
    fecha_alta_parseada = pd.to_datetime(fecha_alta_raw, errors="coerce")
    if pd.isna(fecha_alta_parseada):
        errores.append("fecha_alta_invalida")

    # ── Regla 5: email (opcional, pero si existe debe tener formato válido) ────
    # El patrón [^@\s]+@[^@\s]+\.[^@\s]+ valida que tenga la forma algo@dominio.ext
    email_raw = str(fila.get("email") or "").strip()
    email_valido = False

    if email_raw:
        if re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", email_raw):
            email_valido = True   # Email existe y tiene formato correcto
        else:
            errores.append("email_invalido")  # Email existe pero es inválido

    # ── Regla 6: nro_telefono (opcional, pero si existe debe ser válido) ───────
    # \+? significa "un + opcional". \d{7,15} significa entre 7 y 15 dígitos.
    tel_raw = str(fila.get("nro_telefono") or "").strip()
    tel_valido = False

    if tel_raw:
        if re.fullmatch(r"\+?\d{7,15}", tel_raw):
            tel_valido = True    # Teléfono existe y tiene formato correcto
        else:
            errores.append("telefono_invalido")  # Teléfono existe pero es inválido

    # ── Regla 7: debe tener al menos un método de contacto válido ─────────────
    # Si ninguno de los dos (email o teléfono) está disponible y válido, es error.
    if not email_valido and not tel_valido:
        errores.append("sin_metodo_contacto")

    return errores


def obtener_motivo(fila: pd.Series) -> str:
    """
    Función auxiliar que llama a validar_registro() y convierte la lista
    de errores en un string separado por '; ' para guardarlo en el DataFrame.

    Parámetros
    ----------
    fila : pd.Series
        Una fila del DataFrame de clientes.

    Retorna
    -------
    str
        String con los motivos de error separados por '; '.
        String vacío "" si el registro es válido.
    """
    errores = validar_registro(fila)
    # Usamos VALIDATION_ERRORS para convertir la clave al mensaje descriptivo
    mensajes = [VALIDATION_ERRORS[clave] for clave in errores]
    return "; ".join(mensajes)


print("✅ Funciones de validación definidas correctamente.")

### B4 — Aplicar Validación y Separar DataFrames

Aplicamos `obtener_motivo()` a cada fila del DataFrame usando `.apply()`.  
Esto ejecuta la función fila por fila y almacena el resultado en la nueva
columna `motivo_invalido`.

Luego separamos en dos DataFrames:
- Registros donde `motivo_invalido` es vacío `""` → **válidos**
- Registros donde `motivo_invalido` tiene algún texto → **inválidos**

In [ ]:
# ── Aplicar la función de validación a cada fila ──────────────────────────────
# axis=1 indica que la función se aplica fila por fila (no columna por columna).
df_trabajo["motivo_invalido"] = df_trabajo.apply(obtener_motivo, axis=1)

# ── Crear máscaras booleanas para separar los registros ───────────────────────
# Una máscara es una columna de True/False que usamos como filtro.
mascara_validos   = df_trabajo["motivo_invalido"] == ""
mascara_invalidos = df_trabajo["motivo_invalido"] != ""

# ── Separar en válidos ────────────────────────────────────────────────────────
# drop(columns=...) elimina la columna auxiliar que ya no necesitamos en válidos.
df_validos   = df_trabajo[mascara_validos].drop(columns=["motivo_invalido"]).copy()
df_invalidos = df_trabajo[mascara_invalidos].copy()

print("=== RESULTADO DE VALIDACIÓN ===")
print(f"  ✅ Registros válidos  : {len(df_validos)}")
print(f"  ❌ Registros inválidos: {len(df_invalidos)}")
print(f"  📋 Total procesados   : {len(df_trabajo)}")

In [ ]:
df_invalidos.head(10)

### B5 — Guardado de Registros Inválidos

Guardamos `df_invalidos` en Parquet para que el equipo de calidad de datos
pueda revisarlos y corregirlos en el origen.

También mostramos el detalle para verificar que las validaciones funcionaron correctamente.

In [ ]:
# ── Guardado de inválidos en Parquet ──────────────────────────────────────────
df_invalidos.to_parquet(INVALIDOS_PATH, engine="pyarrow", index=False)
print(f"✅ Inválidos guardados en: {INVALIDOS_PATH}")

# ── Mostrar detalle de los registros inválidos ────────────────────────────────
print("\n=== DETALLE DE REGISTROS INVÁLIDOS ===")
columnas_a_mostrar = ["id_cliente", "nombres", "apellidos", "numero_documento", "motivo_invalido"]
print(df_invalidos[columnas_a_mostrar].to_string(index=False))

---

## 5 — PARTE C: Carga a Delta Table (Silver)

### ¿Qué es una Delta Table?
Una **Delta Table** es un formato de almacenamiento que agrega capacidades ACID
(Atomicidad, Consistencia, Aislamiento, Durabilidad) sobre archivos Parquet.

Características clave:
- Mantiene un **log de transacciones** (`_delta_log/`) con el historial de cada operación.
- Soporta **upsert** (merge): actualiza registros existentes e inserta los nuevos.
- No requiere PySpark: la librería `deltalake` (delta-rs) es Python puro.

### ¿Qué es un MERGE/UPSERT?
Es una operación que combina dos tablas usando una **clave primaria** (en este caso `numero_documento`):
- Si el documento **ya existe** en la tabla → **actualiza** todos sus campos.
- Si el documento **no existe** → **inserta** el registro como nuevo.

Esto permite ejecutar el pipeline múltiples veces sin generar duplicados.

### Resultado esperado
- Primera ejecución: se **crea** la Delta Table con los registros válidos.
- Ejecuciones siguientes: se ejecuta el **merge** actualizando o insertando según corresponda.

In [ ]:
# ── Convertir df_validos a formato PyArrow ────────────────────────────────────
# La librería deltalake trabaja internamente con PyArrow Tables, no con DataFrames.
# preserve_index=False: no incluimos el índice de Pandas en la tabla.
pa_table = pa.Table.from_pandas(df_validos, preserve_index=False)

print(f"=== TABLA PYARROW LISTA ===")
print(f"  Filas    : {pa_table.num_rows}")
print(f"  Columnas : {pa_table.num_columns}")

In [ ]:
# ── Verificar si la Delta Table ya existe ─────────────────────────────────────
# is_deltatable() revisa si la carpeta contiene un directorio _delta_log válido.
tabla_existe = DeltaTable.is_deltatable(DELTA_PATH)
print(f"  Delta Table existe en '{DELTA_PATH}': {tabla_existe}")

if not tabla_existe:
    # ── Primera ejecución: crear la Delta Table ───────────────────────────────
    # mode="overwrite" crea la tabla desde cero con los datos actuales.
    print("\n=== DELTA TABLE NO EXISTE — CREANDO ===")
    write_deltalake(DELTA_PATH, pa_table, mode="overwrite")
    print(f"✅ Delta Table creada en: {DELTA_PATH}")

else:
    # ── Ejecuciones siguientes: MERGE/UPSERT ─────────────────────────────────
    # Cargamos la tabla existente y ejecutamos el merge.
    print("\n=== DELTA TABLE EXISTE — EJECUTANDO MERGE ===")
    dt = DeltaTable(DELTA_PATH)

    # El predicado define la clave de join: numero_documento es nuestra PK.
    # when_matched_update_all()     → si el doc ya existe, actualiza todos sus campos.
    # when_not_matched_insert_all() → si el doc es nuevo, inserta el registro.
    (
        dt.merge(
            source=pa_table,
            predicate="source.numero_documento = target.numero_documento",
            source_alias="source",
            target_alias="target",
        )
        .when_matched_update_all()
        .when_not_matched_insert_all()
        .execute()
    )
    print(f"✅ Merge ejecutado en: {DELTA_PATH}")

# ── Verificación final: contar registros en la tabla ─────────────────────────
dt_final = DeltaTable(DELTA_PATH)
total_registros = dt_final.to_pandas().shape[0]

print(f"\n=== DELTA TABLE — ESTADO FINAL ===")
print(f"  Total registros en Silver : {total_registros}")
print(f"  Versión actual de la tabla: {dt_final.version()}")
print(f"  Delta log en              : {DELTA_PATH}/_delta_log")

---

## 6. Resumen del Pipeline

El pipeline ELT procesó el archivo `clientes_crudos.csv` pasando por tres capas:

| Capa | Formato | Descripción | Ruta |
|------|---------|-------------|------|
| 🥉 **Bronze** | Parquet | Datos crudos con fecha de ingesta, sin transformaciones | `datos/output/bronze_clientes_YYYYMMDD.parquet` |
| ❌ **Inválidos** | Parquet | Registros que no pasaron la validación + motivo detallado | `datos/output/invalidos_clientes_YYYYMMDD.parquet` |
| 🥈 **Silver** | Delta Table | Datos válidos con upsert por `numero_documento` | `datos/silver/clientes_delta/` |

---

### Reglas de validación aplicadas

| # | Campo | Regla |
|---|-------|-------|
| 1 | `nombres` | Sin dígitos, no vacío |
| 2 | `apellidos` | Sin dígitos, no vacío |
| 3 | `numero_documento` | Solo dígitos, 6–8 caracteres, obligatorio |
| 4 | `fecha_alta` | Obligatoria, formato fecha válido |
| 5 | `email` | Opcional; si presente, formato `algo@dominio.ext` |
| 6 | `nro_telefono` | Opcional; si presente, `+` opcional + 7–15 dígitos |
| 7 | Contacto | Al menos un método de contacto válido (email o teléfono) |

> Las columnas `fecha_nacimiento` y `fecha_baja` son opcionales.  
> Si tienen un formato inválido, se reemplazan por `NaT` sin mover el registro a inválidos.